# 🏠 EDA de avisos inmobiliarios: capa Bronze en Databricks

**Proyecto 1 · Bootcamp de Data Engineering** · Databricks · Spark SQL · Unity Catalog · Delta Lake

---

### 🎯 Objetivo
Ingerir un dataset crudo de **~1,25 millones de avisos inmobiliarios** de Buenos Aires (scrapeados de portales como MercadoLibre, Argenprop y Liderprop) en una arquitectura **medallion**, y realizar un **análisis exploratorio (EDA)** sobre la capa *bronze* para diagnosticar la calidad de los datos y definir las reglas de limpieza de la capa *silver*.

### 🏗️ Arquitectura

```
properties_raw.csv
        │  carga manual
        ▼
/Volumes/bootcamp/landing/archivos        ← LANDING  (Volume de Unity Catalog)
        │  read_files() + filtro de URL válida
        ▼
bootcamp.bronze.properties_bronze         ← BRONZE   (tabla Delta, todo como string)
        │  EDA (este notebook)
        ▼
Hallazgos  →  reglas de limpieza para SILVER
```

### 🧰 Técnicas aplicadas
`read_files` · Volumes · CTEs · `unpivot` · window functions (`sum() over()`, `row_number()`) · percentiles (P1/P25/P50/P75/P99) · detección de outliers y duplicados · vistas temporales

### 📑 Contenido
| # | Sección |
|---|---|
| 0 | Setup e ingesta: landing → bronze |
| 1 | Exploración inicial |
| 2 | Valores nulos |
| 3 | Cardinalidad de variables categóricas |
| 4 | Estadísticas de variables numéricas |
| 5 | Calidad de datos: inválidos, duplicados y outliers |
| 6 | Análisis con CTEs y window functions |
| ✅ | Hallazgos y próximos pasos |

---
## 0️⃣ Setup e ingesta: landing → bronze

Preparamos el catálogo y las capas donde vivirán los datos:

| Paso | Acción | Objeto |
|---|---|---|
| 1 | Subir el CSV crudo al Volume | `properties_raw.csv` |
| 2 | Crear catálogo y esquemas | `bootcamp`, `bootcamp.landing`, `bootcamp.bronze` |
| 3 | Crear el Volume *managed* para archivos | `bootcamp.landing.archivos` |
| 4 | Verificar que el archivo esté en el Volume | `LIST '/Volumes/...'` |
| 5 | Crear la tabla bronze desde el CSV | `bootcamp.bronze.properties_bronze` |

> En la carga a bronze solo se conservan las filas con una `url` que empieza por `https`: es el filtro mínimo para descartar líneas corruptas del CSV.

In [0]:
%sql
-- ============================================================
-- PASO 2: Crear el esquema Landing
-- ============================================================
-- Crear catalogo si no existe
Create catalog if not exists bootcamp; --ya existia es como un ecositema
-- Usar tu catálogo
USE CATALOG bootcamp; -- estos son los datasets
-- Crear esquema Landing
CREATE SCHEMA IF NOT EXISTS bootcamp.landing
COMMENT 'Esquema para archivos';
-- Crear esquema Raw
CREATE SCHEMA IF NOT EXISTS bootcamp.bronze
COMMENT 'Esquema para datos crudos sin procesar';
-- Verificar
SHOW SCHEMAS;

databaseName
bronze
default
information_schema
landing
practice


In [0]:
%sql
-- ============================================================
-- PASO 3: Crear the Volume para archivos
-- ============================================================
-- Crear volume tipo MANAGED (Databricks administra el storage)
CREATE VOLUME IF NOT EXISTS bootcamp.landing.archivos
COMMENT 'Volume para almacenar archivos CSV crudos';
-- Verificar que se creó
SHOW VOLUMES IN bootcamp.landing;

database,volume_name
landing,archivos


In [0]:
%sql
-- ============================================================
-- PASO 4: Verificar que el archivo se subió correctamente
-- ============================================================
-- Listar archivos en el volume
LIST '/Volumes/bootcamp/landing/archivos/';


path,name,size,modification_time
/Volumes/bootcamp/landing/archivos/properties_raw.csv,properties_raw.csv,368203169,1793466503000


In [0]:
%sql
-- ============================================================
-- PASO 5: Crear la tabla properties_bronze
-- ============================================================
-- Primero, eliminamos la tabla si existe (para poder recrearla)
DROP TABLE IF EXISTS bootcamp.bronze.properties_bronze;
-- Crear tabla EXTERNA leyendo el CSV y nos quedamos solo con los registros que tienen url válida
CREATE TABLE bootcamp.bronze.properties_bronze
 SELECT * FROM read_files(
 '/Volumes/bootcamp/landing/archivos/properties_raw.csv',
 format => 'csv',
 header => true
 )
 where url like 'https%' --Filtramos por aquellos links que sean consistentes
 ;

In [0]:
%sql
-- Verificar que la tabla se creó correctamente
SHOW TABLES IN bootcamp.bronze;

database,tableName,isTemporary
bronze,properties_bronze,false


---
### 📊 Diccionario de datos

| Columna | Descripción esperada |
|---|---|
| `id` | ID único de la propiedad |
| `ubicacion` | Dirección completa |
| `numero`, `calle` | Componentes de la dirección |
| `precio`, `expensas` | Valores monetarios |
| `tipo_de_operacion` | alquiler, venta, alquiler_temporario |
| `moneda` | USD o ARS |
| `ambientes` | Cantidad de ambientes |
| `metros_cuadrados_totales/cubiertos` | Superficies |
| `orientacion_cardinal/inmueble` | Norte/Sur, Frente/Contrafrente |
| `piso`, `cochera`, `antiguedad` | Características |
| `estado`, `tipo_vendedor` | Categorías |
| `url` | URL del aviso original |
| `zona`, `fecha`, `hora` | Metadata del scraping |

---
## 1️⃣ Exploración inicial

Primer contacto con la tabla: volumen, esquema y una muestra de registros.

In [0]:
%sql
-- ============================================================
-- EDA 1.1: ¿Cuántos registros tenemos?
-- ============================================================

SELECT COUNT(*) as total_registros
FROM bootcamp.bronze.properties_bronze;

total_registros
1246131


In [0]:
%sql
-- ============================================================
-- EDA 1.2: Explorar estructura de la tabla
-- ============================================================

DESCRIBE TABLE bootcamp.bronze.properties_bronze

col_name,data_type,comment
id,double,null
ubicacion,string,null
precio,string,null
numero,string,null
calle,string,null
expensas,string,null
tipo_de_operacion,string,null
moneda,string,null
ambientes,string,null
metros_cuadrados_totales,string,null


In [0]:
%sql
-- ============================================================
-- EDA 1.3: Ver muestra de datos (columnas principales)
-- ============================================================

select * from bootcamp.bronze.properties_bronze limit 10

id,ubicacion,precio,numero,calle,expensas,tipo_de_operacion,moneda,ambientes,metros_cuadrados_totales,metros_cuadrados_cubiertos,orientacion_cardinal,orientacion_inmueble,piso,cochera,antiguedad,estado,tipo_vendedor,url,zona,fecha,hora,_rescued_data
1156185.0,"Ing. Marconi 3800, Béccar, San Isidro",320000.0,3800.0,Ing. Marconi,null,venta,USD,5.0,288.0,null,null,null,null,1,999.0,a_refaccionar,null,https://www.zonaprop.com.ar/propiedades/clasificado/veclcain-venta-casa-5-amb-cochera-y-jardin-la-horqueta-57817077.html?n_src=Listado&n_pills=Terraza&n_pg=3&n_pos=20,san-isidro,2025-12-04,13:15:22,null
1156186.0,"Amancio Alcorta 600, Villa Adelina, San Isidro",175000.0,600.0,Amancio Alcorta,null,venta,USD,4.0,338.0,null,null,null,null,null,999.0,excelente,null,https://www.zonaprop.com.ar/propiedades/clasificado/veclphin-ph-tipo-casa-4-amb.-con-jardin-villa-adelina-57809607.html?n_src=Listado&n_pills=Apto+cr%C3%A9dito&n_pg=3&n_pos=21,san-isidro,2025-12-04,13:15:22,null
1156187.0,"Alsina 2500, San Isidro, San Isidro",2500000.0,2500.0,Alsina,null,venta,USD,8.0,2138.0,null,null,null,null,1,999.0,excelente,null,https://www.zonaprop.com.ar/propiedades/clasificado/veclcain-venta-casa-san-isidro-las-lomas-57816331.html?n_src=Listado&n_pills=Terraza&n_pg=3&n_pos=22,san-isidro,2025-12-04,13:15:22,null
1156188.0,"Cazon 1100. La Horqueta, San Isidro., La Horqueta, San Isidro",435000.0,1100.0,Cazon,null,venta,USD,5.0,410.0,232.0,null,null,null,1,999.0,null,null,https://www.zonaprop.com.ar/propiedades/clasificado/veclcain-casa-en-venta-y-alquiler-en-la-horqueta-con-jardin-y-57811192.html?n_src=Listado&n_pills=Pileta&n_pg=3&n_pos=23,san-isidro,2025-12-04,13:15:22,null
1156189.0,"Geronimo Barbosa 1300, San Isidro, San Isidro",399000.0,1300.0,Geronimo Barbosa,null,venta,USD,4.0,286.0,200.0,null,null,null,1,999.0,excelente,null,https://www.zonaprop.com.ar/propiedades/clasificado/veclcain-casa-venta-san-isidro-jardin-pileta-apto-credito-57817288.html?n_src=Listado&n_pills=Pileta&n_pg=3&n_pos=24,san-isidro,2025-12-04,13:15:22,null
1156190.0,"Intendente Neyer 3000, Béccar, San Isidro",675000.0,3000.0,Intendente Neyer,null,venta,USD,6.0,1008.0,530.0,null,null,null,1,999.0,excelente,null,https://www.zonaprop.com.ar/propiedades/clasificado/veclcain-espectacular-casa-de-2-plantas-en-venta-san-isidro-de-57811167.html?n_src=Listado&n_pills=Dormitorio+en+suite&n_pg=3&n_pos=25,san-isidro,2025-12-04,13:15:22,null
1156235.0,"Roberto Caamaño 1100, Villa Rosa, Pilar",139000.0,1100.0,Roberto Caamaño,200000.0,venta,USD,2.0,55.0,null,null,null,4.0,1,999.0,excelente,null,https://www.zonaprop.com.ar/propiedades/clasificado/veclapin-venta-depto.-2-amb-cochera-amenities-pilar-km46-57780859.html?n_src=Listado&n_pills=Terraza&n_pg=2&n_pos=24,pilar,2025-12-04,13:15:22,null
1156236.0,"Chacabuco 300, Pilar",145000.0,300.0,Chacabuco,null,venta,USD,2.0,73.0,41.0,null,null,0.0,1,999.0,null,null,https://www.zonaprop.com.ar/propiedades/clasificado/veclapin-departamento-en-venta-luciernagas-2-amb-c-jardin-57782583.html?n_src=Listado&n_pills=Parrilla&n_pg=2&n_pos=25,pilar,2025-12-04,13:15:22,null
1156256.0,"Av. San Martin 1000, Vicente López, Vicente López",500000.0,1000.0,Av. San Martin,null,venta,USD,6.0,229.0,205.0,null,null,null,1,999.0,null,inmobiliaria,https://www.zonaprop.com.ar/propiedades/clasificado/veclcain-casa-4-dorm.-venta-barrio-vicente-lopez-57814948.html?n_src=Listado&n_pills=Parrilla&n_pg=3&n_pos=18,vicente-lopez,2025-12-04,13:15:22,null
1156191.0,"O'Higgins 583, San Isidro, San Isidro",245000.0,583.0,O'Higgins,null,venta,USD,4.0,158.0,153.0,null,null,null,1,999.0,null,null,https://www.zonaprop.com.ar/propiedades/clasificado/veclcapa-dueno-directo-casa-3-dorm.-espectacular-terraza-y-57816058.html?n_src=Listado&n_pills=Quincho&n_pg=3&n_pos=26,san-isidro,2025-12-04,13:15:22,null


---
## 2️⃣ Valores nulos

Identificar campos vacíos es **crítico** para planificar la limpieza.
Se calcula el conteo de nulos de cada columna con `count(*) - count(col)`, se transpone el resultado con `unpivot` (una fila por columna) y se filtran las columnas con **más del 50 % de nulos**, candidatas a eliminarse o tratarse de forma especial.

In [0]:
%sql
-- ============================================================
-- EDA 2.1: Contar valores nulos por columna
-- ============================================================
-- Usamos CTE para calcular el total y los nulos por columna

with nulos as (
    select
        count(*)                                     as total_filas,
        count(*) - count(id)                         as nulo_id,
        count(*) - count(ubicacion)                  as nulo_ubicacion,
        count(*) - count(precio)                     as nulo_precio,
        count(*) - count(numero)                     as nulo_numero,
        count(*) - count(calle)                      as nulo_calle,
        count(*) - count(expensas)                   as nulo_expensas,
        count(*) - count(tipo_de_operacion)          as nulo_tipo_de_operacion,
        count(*) - count(moneda)                     as nulo_moneda,
        count(*) - count(ambientes)                  as nulo_ambientes,
        count(*) - count(metros_cuadrados_totales)   as nulo_metros_cuadrados_totales,
        count(*) - count(metros_cuadrados_cubiertos) as nulo_metros_cuadrados_cubiertos,
        count(*) - count(orientacion_cardinal)       as nulo_orientacion_cardinal,
        count(*) - count(orientacion_inmueble)       as nulo_orientacion_inmueble,
        count(*) - count(piso)                       as nulo_piso,
        count(*) - count(cochera)                    as nulo_cochera,
        count(*) - count(antiguedad)                 as nulo_antiguedad,
        count(*) - count(estado)                     as nulo_estado,
        count(*) - count(tipo_vendedor)              as nulo_tipo_vendedor,
        count(*) - count(url)                        as nulo_url,
        count(*) - count(zona)                       as nulo_zona,
        count(*) - count(fecha)                      as nulo_fecha,
        count(*) - count(hora)                       as nulo_hora,
        count(*) - count(_rescued_data)              as nulo_rescued_data
    from bootcamp.bronze.properties_bronze
), t2 as (
-- ============================================================
-- EDA 2.2: Porcentaje de nulos en columnas clave
-- ============================================================
select
    replace(columna, 'nulo_', '')           as columna,
    nulos,
    round(100 * nulos / total_filas, 2)     as pct_nulos
from nulos
-- ============================================================
-- EDA 2.3: Identificar columnas con >50% de valores nulos
-- ============================================================
-- Columnas con >50% nulos son críticas: hay que decidir si eliminarlas o imputarlas
unpivot (
    nulos for columna in (
        nulo_id, nulo_ubicacion, nulo_precio, nulo_numero, nulo_calle,
        nulo_expensas, nulo_tipo_de_operacion, nulo_moneda, nulo_ambientes,
        nulo_metros_cuadrados_totales, nulo_metros_cuadrados_cubiertos,
        nulo_orientacion_cardinal, nulo_orientacion_inmueble, nulo_piso,
        nulo_cochera, nulo_antiguedad, nulo_estado, nulo_tipo_vendedor,
        nulo_url, nulo_zona, nulo_fecha, nulo_hora, nulo_rescued_data
    )
)
)

select * from t2 where pct_nulos > 50
order by pct_nulos desc

columna,nulos,pct_nulos
rescued_data,1246131,100.0
orientacion_cardinal,1152663,92.5
tipo_vendedor,1054326,84.61
orientacion_inmueble,1041962,83.62
piso,958638,76.93
cochera,743929,59.7
estado,736825,59.13
expensas,680182,54.58


---
## 3️⃣ Cardinalidad de variables categóricas

¿Qué valores toman las columnas categóricas y con qué frecuencia?
El porcentaje sobre el total se calcula con la window function `sum(count(*)) over()`, sin necesidad de subconsultas.

In [0]:
%sql
-- ============================================================
-- EDA 3.1: Distribución de tipo_de_operacion
-- ============================================================

select 
tipo_de_operacion,
count(*) as total,
round(100 * count(*) / (select count(*) from bootcamp.bronze.properties_bronze),2) as pct_total
from bootcamp.bronze.properties_bronze
group by all
order by total desc

tipo_de_operacion,total,pct_total
alquiler,626540,50.28
venta,603346,48.42
alquiler_temporal,14524,1.17
null,1400,0.11
alquiler_de_habitacion,206,0.02
alquiler_anual,35,0.0
venta/alquiler,19,0.0
alquiler_temporario,13,0.0
venta_y_alquiler,9,0.0
consultar,6,0.0


In [0]:
%sql
-- ============================================================
-- EDA 3.2: Distribución de moneda
-- ============================================================

select 
moneda,
count(*) as total,
round(100 * count(*) / sum(count(*)) over(),2) as pct_total
from bootcamp.bronze.properties_bronze
group by all
order by total desc

moneda,total,pct_total
USD,851579,68.34
ARS,388893,31.21
null,5643,0.45
MXN,8,0.0
consultar,2,0.0
guaranies,2,0.0
uyu,2,0.0
null,1,0.0
ar,1,0.0


In [0]:
%sql
-- ============================================================
-- EDA 3.3: Distribución de ambientes
-- ============================================================
select 
ambientes,
count(*) as total,
round( count(*) * 100 / sum( count(*) ) over() ,2) as pct_total
from bootcamp.bronze.properties_bronze
group by all
order by total desc

ambientes,total,pct_total
2.0,343223,27.54
3.0,326635,26.21
4.0,218585,17.54
1.0,156331,12.55
5.0,104737,8.4
6.0,46063,3.7
7.0,20143,1.62
null,13566,1.09
8.0,7669,0.62
10.0,2655,0.21


In [0]:
%sql
-- ============================================================
-- EDA 3.4: Top 15 zonas con más propiedades
-- ============================================================
select 
zona,
count(*) as total,
round( count(*) * 100 / sum( count(*) ) over() ,2) as pct_total
from bootcamp.bronze.properties_bronze
group by all
order by total desc
limit 15

zona,total,pct_total
capital-federal,450364,36.14
tigre,91470,7.34
pilar,80214,6.44
vicente-lopez,67555,5.42
escobar,57302,4.6
san-isidro,47597,3.82
general-san-martin,42987,3.45
san-miguel,30786,2.47
san-fernando,30643,2.46
jose-c-paz,16576,1.33


In [0]:
%sql
-- ============================================================
-- EDA 3.5: Distribución de estado de las propiedades
-- ============================================================

select 
estado,
count(*) as total,
round( count(*) * 100 / sum( count(*) ) over() ,2) as pct_total
from bootcamp.bronze.properties_bronze
group by all
order by total desc
--limit 15

estado,total,pct_total
null,736825,59.13
excelente,370092,29.7
bueno,109399,8.78
a_refaccionar,26521,2.13
muy bueno,1076,0.09
a_estrenar,849,0.07
a estrenar,362,0.03
impecable,297,0.02
0,191,0.02
terminada,124,0.01


---
## 4️⃣ Estadísticas de variables numéricas

Como en bronze todo es `string`, primero se construyen dos **vistas temporales**:

| Vista | Qué hace |
|---|---|
| `propiedades_clean` | Castea a `double` precio, ambientes, m² y antigüedad. Los valores con letras (`NaN`, texto) pasan a `null` |
| `propiedades_clean_2` | Se queda solo con **USD/ARS** y **venta/alquiler**, y elimina outliers de precio fuera del rango **P1–P99** de cada combinación moneda × operación |

Las estadísticas de precio siempre se separan por **moneda y tipo de operación**: mezclar dólares con pesos, o ventas con alquileres, produce promedios sin sentido.

In [0]:
%sql
create or replace temporary View propiedades_clean as
select 
CASE
  WHEN precio RLIKE '^[^a-zA-Z]+$' THEN precio::double
  ELSE NULL
  END as precio,
  moneda,
CASE
  WHEN ambientes RLIKE '^[^a-zA-Z]+$' THEN ambientes::double
  ELSE NULL
END as ambientes
 ,
 CASE
  WHEN metros_cuadrados_totales RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_totales::double
  ELSE NULL
END as metros_cuadrados_totales
 ,
 CASE
  WHEN metros_cuadrados_cubiertos RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_cubiertos::double
  ELSE NULL
END as metros_cuadrados_cubiertos
 ,
 CASE
  WHEN antiguedad RLIKE '^[^a-zA-Z]+$' THEN antiguedad::double
  ELSE NULL
END as antiguedad,
tipo_de_operacion
,id
,ubicacion
,numero
,calle
,expensas
,orientacion_cardinal
,orientacion_inmueble
,piso
,cochera
,estado
,tipo_vendedor
,url
,zona
,fecha
,hora
from bootcamp.bronze.properties_bronze

In [0]:
%sql

-- ============================================================
-- EDA 4.1: Estadísticas de PRECIO (separado por moneda y tipo de operación)
-- ============================================================

SELECT 
    moneda,
    tipo_de_operacion,
    COUNT(*) as cantidad,
    ROUND(MIN(precio), 2) as precio_min,
    ROUND(MAX(precio), 2) as precio_max,
    ROUND(AVG(precio), 2) as precio_promedio,
    ROUND(PERCENTILE(precio, 0.5), 2) as precio_mediana,
    ROUND(PERCENTILE(precio, 0.25), 2) as precio_p25,
    ROUND(PERCENTILE(precio, 0.75), 2) as precio_p75
FROM propiedades_clean
WHERE precio IS NOT NULL AND precio > 0
GROUP BY moneda, tipo_de_operacion
ORDER BY cantidad DESC;


moneda,tipo_de_operacion,cantidad,precio_min,precio_max,precio_promedio,precio_mediana,precio_p25,precio_p75
USD,venta,589016,0.06,1.111111111E9,219097.52,135000.0,79500.0,241200.0
ARS,alquiler,374379,1.0,1.2E9,793701.4,650000.0,500000.0,900000.0
USD,alquiler,248213,1.0,2.14423E7,18585.05,1600.0,1000.0,3000.0
USD,alquiler_temporal,12342,1.0,4200000.0,8147.83,1300.0,700.0,3000.0
ARS,venta,9566,1.0,1.434768228E9,9169126.52,700000.0,240000.0,1400000.0
ARS,alquiler_temporal,1999,65.0,4000000.0,697957.78,690000.0,500000.0,790000.0
USD,null,857,35.0,2000000.0,139784.69,78000.0,3700.0,160000.0
null,venta,812,1.0,5000000.0,257261.63,121600.0,75000.0,216004.5
null,alquiler,775,1.0,1.1111111E7,474931.97,75000.0,1350.0,700000.0
ARS,null,486,100.0,5000000.0,872479.21,730000.0,500000.0,1000000.0


In [0]:
%sql
-- ============================================================
-- EDA 4.1b (BONUS — no está en el PDF): Estadísticas de PRECIO (filtrado)
-- Solo USD/ARS, operaciones principales, sin outliers (P1-P99)
-- ============================================================
    WITH limites AS (
    SELECT 
      moneda,
      tipo_de_operacion,
      PERCENTILE(precio, 0.01) AS p01,
      PERCENTILE(precio, 0.99) AS p99
    FROM propiedades_clean
    WHERE precio > 0
      AND moneda IN ('USD', 'ARS')
      AND tipo_de_operacion IN ('venta', 'alquiler')
    GROUP BY moneda, tipo_de_operacion
  )
  SELECT 
    p.moneda,
    p.tipo_de_operacion,
    COUNT(*) AS cantidad,
    ROUND(MIN(p.precio), 2) AS precio_min,
    ROUND(MAX(p.precio), 2) AS precio_max,
    ROUND(AVG(p.precio), 2) AS precio_promedio,
    ROUND(PERCENTILE(p.precio, 0.5), 2) AS precio_mediana,
    ROUND(PERCENTILE(p.precio, 0.25), 2) AS precio_p25,
    ROUND(PERCENTILE(p.precio, 0.75), 2) AS precio_p75
  FROM propiedades_clean p
  JOIN limites l 
    ON p.moneda = l.moneda 
    AND p.tipo_de_operacion = l.tipo_de_operacion
  WHERE p.precio BETWEEN l.p01 AND l.p99
  GROUP BY p.moneda, p.tipo_de_operacion
  ORDER BY cantidad DESC

moneda,tipo_de_operacion,cantidad,precio_min,precio_max,precio_promedio,precio_mediana,precio_p25,precio_p75
USD,venta,577278,2300.0,1515000.0,196176.52,135000.0,80000.0,240000.0
ARS,alquiler,367201,160000.0,2800000.0,760959.57,650000.0,500000.0,900000.0
USD,alquiler,244038,400.0,550000.0,5235.92,1600.0,1000.0,2950.0
ARS,venta,9394,1200.0,1.2E8,1513914.95,700000.0,250000.0,1400000.0


In [0]:
%sql
-- ============================================================
-- Creamos la nueva vista temporal
-- ============================================================
CREATE OR REPLACE TEMPORARY VIEW propiedades_clean_2 AS
(
    WITH limites AS (
    SELECT 
      moneda,
      tipo_de_operacion,
      PERCENTILE(precio, 0.01) AS p01,
      PERCENTILE(precio, 0.99) AS p99
    FROM propiedades_clean
    WHERE precio > 0
      AND moneda IN ('USD', 'ARS')
      AND tipo_de_operacion IN ('venta', 'alquiler')
    GROUP BY moneda, tipo_de_operacion
  )
  SELECT p.*
  FROM propiedades_clean p
  JOIN limites l 
    ON p.moneda = l.moneda 
    AND p.tipo_de_operacion = l.tipo_de_operacion
  WHERE p.precio BETWEEN l.p01 AND l.p99
);

In [0]:
%sql
-- ============================================================
-- EDA 4.2: Estadísticas de METROS CUADRADOS
-- ============================================================

SELECT 
    'metros_cuadrados_totales' as columna,
    COUNT(*) as cantidad_no_nulos,
    ROUND(MIN(metros_cuadrados_totales), 2) as minimo,
    ROUND(MAX(metros_cuadrados_totales), 2) as maximo,
    ROUND(AVG(metros_cuadrados_totales), 2) as promedio,
    ROUND(PERCENTILE(metros_cuadrados_totales, 0.5), 2) as mediana
FROM propiedades_clean_2
WHERE metros_cuadrados_totales IS NOT NULL AND metros_cuadrados_totales > 0

UNION ALL

SELECT 
    'metros_cuadrados_cubiertos' as columna,
    COUNT(*) as cantidad_no_nulos,
    ROUND(MIN(metros_cuadrados_cubiertos), 2) as minimo,
    ROUND(MAX(metros_cuadrados_cubiertos), 2) as maximo,
    ROUND(AVG(metros_cuadrados_cubiertos), 2) as promedio,
    ROUND(PERCENTILE(metros_cuadrados_cubiertos, 0.5), 2) as mediana
FROM propiedades_clean_2
WHERE metros_cuadrados_cubiertos IS NOT NULL AND metros_cuadrados_cubiertos > 0;



columna,cantidad_no_nulos,minimo,maximo,promedio,mediana
metros_cuadrados_totales,1149772,1.0,2.147483647E9,9107.14,78.0
metros_cuadrados_cubiertos,889644,1.0,2.0E9,6879.64,74.0


---
## 5️⃣ Calidad de datos: inválidos, duplicados y outliers

| Chequeo | Pregunta |
|---|---|
| 5.1 | ¿Cuántos registros tienen valores inválidos o centinela (ej. `antiguedad = 999`)? |
| 5.2 – 5.3 | ¿Cuántos avisos están duplicados (misma `url` y `precio`)? |
| 5.4 | ¿Cuántos precios caen por debajo de P1 o por encima de P99? |

In [0]:
%sql
-- ============================================================
-- EDA 5.1: Reporte de calidad de datos usando CTEs
-- ============================================================

select 
count(*) as total,

count_if(precio is null or precio <=0) as precio_invalido,
round ( 100 * count_if(precio is null or precio <=0) / count(*), 2) as pct_precio_invalido,

count_if(metros_cuadrados_totales is null or metros_cuadrados_totales <=0) as metros_totales_invalidos,
round ( 100 * count_if(metros_cuadrados_totales is null or metros_cuadrados_totales <=0) / count(*), 2) as pct_pmetros_cuadrados_totales,

count_if(antiguedad = 999 ) as antiguedad_invalidad,
round ( 100 * count_if(antiguedad = 999 ) / count(*), 2) as pct_antiguedad,

count_if(ambientes is null or precio <=0) as ambientes_invalidos,
round ( 100 * count_if(ambientes is null or ambientes <=0) / count(*), 2) as pct_ambintes,

count_if(moneda is null) as moneda_invalidos,
round ( 100 * count_if(moneda is null) / count(*), 2) as pct_moneda,

count_if(tipo_de_operacion is null) as moneda_invalidos,
round ( 100 * count_if(tipo_de_operacion is null) / count(*), 2) as pct_precio_invalido

from propiedades_clean_2

total,precio_invalido,pct_precio_invalido,metros_totales_invalidos,pct_pmetros_cuadrados_totales,antiguedad_invalidad,pct_antiguedad,ambientes_invalidos,pct_ambintes,moneda_invalidos,pct_moneda,moneda_invalidos,pct_precio_invalido
1197911,0,0.0,48139,4.02,1190190,99.36,12695,1.24,0,0.0,0,0.0


In [0]:
%sql
with t1 as (
select 
url as grupo,
precio,
count(*) as total
from propiedades_clean_2
group by all
having total >1
order by total desc
)
select 
count(grupo) as cantidad_grupos,
sum(total) as total_duplicados,
sum(total - 1) as total_extras --recuerda que total es la cantidad de duplicado si le quitas 1 que es la cantidad correcta, sumamra todos lo que están extra
from t1

cantidad_grupos,total_duplicados,total_extras
81114,211864,130750


In [0]:
%sql
-- ============================================================
-- EDA 5.3: Ver ejemplos de duplicados
-- ============================================================
select 
url as grupo,
precio,
count(*) as total
from propiedades_clean_2
group by all
having total >1
order by total desc
limit 10

grupo,precio,total
https://departamento.mercadolibre.com.ar/MLA-2059467746-venta-departamento-2-ambientes-con-cochera-a-estrenar-en-villa-luzuriaga-uf-4-_JM,62000.0,12
https://departamento.mercadolibre.com.ar/MLA-1507670893-venta-departamento-3-ambientes-lateral-en-ramos-mejia-_JM,56000.0,11
https://www.argenprop.com/casa-en-venta-en-hurlingham-3-ambientes--16120684,155000.0,10
https://liderprop.com/es-ar/propiedades/5509769/alquiler--casa--muniz/,900000.0,10
https://departamento.mercadolibre.com.ar/MLA-1487887695-venta-departamento-2-ambientes-en-lomas-del-mirador-2a-_JM,69000.0,10
https://www.argenprop.com/departamento-en-alquiler-en-moron-sur-1-ambiente--17784947,250000.0,10
https://www.argenprop.com/casa-en-venta-en-domselaar-2-ambientes--17286651,21000.0,10
https://www.argenprop.com/departamento-en-alquiler-en-ramos-mejia-1-ambiente--17626376,300000.0,10
https://liderprop.com/es-ar/propiedades/5864972/venta--casa--hurlingham-hurlingham/,95000.0,10
https://www.argenprop.com/departamento-en-alquiler-en-wilde-1-ambiente--16853930,350000.0,10


In [0]:
%sql
-- ============================================================
-- EDA 5.4: Detectar outliers extremos en precio
-- ============================================================

-- Propiedades con precios sospechosamente altos o bajos
WITH stats AS (
    SELECT 
        moneda,
        PERCENTILE(precio, 0.01) as p01,
        PERCENTILE(precio, 0.99) as p99
    FROM propiedades_clean_2
    WHERE precio > 0
    GROUP BY moneda
)
SELECT 
    p.moneda,
    'Muy bajo (< P01)' as tipo_outlier,
    COUNT(*) as cantidad
FROM propiedades_clean_2 p
JOIN stats s ON p.moneda = s.moneda
WHERE p.precio < s.p01 AND p.precio > 0
GROUP BY p.moneda

UNION ALL

SELECT 
    p.moneda,
    'Muy alto (> P99)' as tipo_outlier,
    COUNT(*) as cantidad
FROM propiedades_clean_2 p
JOIN stats s ON p.moneda = s.moneda
WHERE p.precio > s.p99
GROUP BY p.moneda
ORDER BY moneda, tipo_outlier;

### 🧹 Vista `bronze_EDA`: primer borrador de limpieza

Antes de construir la capa silver, se prueba la lógica de limpieza en una vista temporal:

- **Numéricos:** `precio`, `numero`, `expensas`, `ambientes` y m² se validan con expresiones regulares y rangos; si no son números válidos, pasan a `null`.
- **Moneda:** se estandarizan variantes (`dolares`, `us`, `pesos`, `ars`) a códigos ISO (`USD`, `ARS`).
- **Cochera:** se convierte a un flag (`tiene` → 1).

In [0]:
%sql
create or replace temp view bronze_EDA as (
SELECT 
    edl.id,
    edl.ubicacion,
    CASE 
        WHEN edl.precio = 'NaN' OR edl.precio NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        WHEN edl.precio::float BETWEEN -2147483648 AND 2147483648 THEN edl.precio::float
        ELSE NULL
    END AS precio,
    CASE 
        WHEN edl.numero = 'NaN' OR edl.numero NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        WHEN edl.numero::float BETWEEN -10000 AND 50000 THEN edl.numero::float
        ELSE NULL
    END AS numero,
    edl.calle,
    CASE
        WHEN edl.expensas = 'NaN' OR edl.expensas NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        WHEN edl.expensas::float BETWEEN -20000000 AND 20000000 THEN edl.expensas::float
        ELSE NULL
    END AS expensas,
    edl.tipo_de_operacion,
    CASE
        WHEN lower(edl.moneda) LIKE '%dolares%' THEN 'USD'
        WHEN lower(edl.moneda) LIKE '%us%' THEN 'USD'
        WHEN lower(edl.moneda) LIKE '%mxn%' THEN 'MXN'
        WHEN lower(edl.moneda) LIKE '%pesos%' THEN 'ARS'
        WHEN lower(edl.moneda) LIKE '%ars%' THEN 'ARS'
        ELSE edl.moneda
    END AS moneda,
    CASE
        WHEN edl.ambientes = 'NaN' OR edl.ambientes NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        ELSE edl.ambientes::float
    END AS ambientes,
    CASE 
        WHEN edl.metros_cuadrados_totales = 'NaN' OR edl.metros_cuadrados_totales NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        ELSE edl.metros_cuadrados_totales::decimal
    END AS metros_cuadrados_totales,
    CASE 
        WHEN edl.metros_cuadrados_cubiertos = 'NaN' OR edl.metros_cuadrados_cubiertos NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        ELSE edl.metros_cuadrados_cubiertos::decimal
    END AS metros_cuadrados_cubiertos,
    edl.orientacion_cardinal,
    edl.orientacion_inmueble,
    CASE
        WHEN edl.piso IS NULL THEN NULL
        WHEN edl.piso = 'NaN' THEN NULL
        ELSE edl.piso::float
    END AS piso,
    CASE 
        WHEN edl.cochera = 'tiene' THEN 1
        ELSE NULL
    END AS cochera,
    edl.antiguedad,
    edl.estado,
    edl.tipo_vendedor,
    edl.url,
    edl.zona,
    edl.fecha,
    edl.hora
FROM propiedades_clean_2 edl
);

---
## 6️⃣ Análisis con CTEs y window functions

Ranking de zonas por precio promedio con `row_number() over (order by avg(precio) desc)`, y comparación de cada zona contra el promedio global.

> ⚠️ Estas consultas promedian juntas ambas monedas y tipos de operación, así que sirven como ejercicio técnico de window functions. Su interpretación y limitaciones se discuten en los hallazgos.

In [0]:
%sql
-- ============================================================
-- EDA 6.1: Precio promedio por zona con RANKING (ROW_NUMBER)
-- Usar CTEs para calcular y rankear zonas por precio
-- ============================================================ 
select
zona,
count(zona) as cant_prop,
round(avg(precio),2) as promedio,
row_number() over (order by avg(precio) desc) as ranking
from propiedades_clean_2 
group by all
order by promedio desc

zona,cant_prop,promedio,ranking
gba-zona-sur--florencio-varela,645,1144657.02,1
bsas-gba-sur/florencio-varela,1050,725162.7,2
bsas-gba-oeste/general-rodriguez,609,718033.2,3
gba-zona-norte--san-isidro,3373,450523.07,4
vicente-lopez,64724,444562.95,5
capital-federal,431023,435961.74,6
san-isidro,45687,417339.44,7
bsas-gba-norte/vicente-lopez,4604,410309.42,8
bsas-gba-norte/san-isidro,4028,384622.21,9
gba-zona-norte--tigre,8340,357636.8,10


In [0]:
%sql
-- ============================================================
-- EDA 6.2: Comparar precio por zona con el promedio general
-- Usar AVG() OVER() para calcular promedio global sin agrupar
-- ============================================================
select
zona,
count(zona) as cant_prop,
round(avg(precio),2) as promedio_zona,
round((select avg(precio) from propiedades_clean_2),2) as promedio_global,
row_number() over (order by avg(precio) desc) as ranking
from propiedades_clean_2 
group by all
order by promedio_zona desc

zona,cant_prop,promedio_zona,promedio_global,ranking
gba-zona-sur--florencio-varela,645,1144657.02,340737.32,1
bsas-gba-sur/florencio-varela,1050,725162.7,340737.32,2
bsas-gba-oeste/general-rodriguez,609,718033.2,340737.32,3
gba-zona-norte--san-isidro,3373,450523.07,340737.32,4
vicente-lopez,64724,444562.95,340737.32,5
capital-federal,431023,435961.74,340737.32,6
san-isidro,45687,417339.44,340737.32,7
bsas-gba-norte/vicente-lopez,4604,410309.42,340737.32,8
bsas-gba-norte/san-isidro,4028,384622.21,340737.32,9
gba-zona-norte--tigre,8340,357636.8,340737.32,10


---
## ✅ Hallazgos y próximos pasos

### 1. Volumen y estructura *(EDA 1.1 – 1.2)*
- **1.246.131 avisos inmobiliarios** cargados en bronze desde el CSV, filtrando solo las filas con URL válida.
- Todas las columnas llegaron como `string`, así que hace falta tipar en silver.

### 2. Valores nulos *(EDA 2.1 – 2.3)*
8 columnas tienen más del 50 % de nulos:

| Columna | % nulos | Decisión sugerida |
|---|---|---|
| _rescued_data | 100 % | esperado: no hubo errores de parseo |
| orientacion_cardinal | 92,50 % | eliminar |
| tipo_vendedor | 84,61 % | eliminar |
| orientacion_inmueble | 83,62 % | eliminar |
| piso | 76,93 % | conservar solo para departamentos |
| cochera | 59,70 % | nulo = "no tiene" (flag 0/1) |
| estado | 59,13 % | conservar como categoría "sin dato" |
| expensas | 54,58 % | conservar, sin imputar |

### 3. Variables categóricas *(EDA 3.1 – 3.5)*
- **`tipo_de_operacion`:** 28 valores distintos. Alquiler (50,28 %) y venta (48,42 %) concentran el 98,7 %. El resto son typos (`alquier`, `alguilar`, `alquieler`), sinónimos (`alquiler_temporal`, `temporario`, `alquiler_de_temporada`) y valores mixtos (`venta/alquiler`, `venta y alquiler`, `alquiler/venta`). Hace falta un mapeo a pocas categorías.
- **`moneda`:** USD 68,34 %, ARS 31,21 % y nulos 0,45 %. Los valores residuales (`MXN`, `guaranies`, `uyu`, `ar`, `consultar`) suman 15 registros.
- **`ambientes`:** 2 y 3 ambientes suman el 53,75 %. Hay valores anómalos: 2.255 avisos con `0`, valores no enteros (`1.5`, `2.5`) y casos de 33, 50 o 66 ambientes.
- **`zona`:** Capital Federal concentra el **36,14 %** de los avisos, seguida por Tigre (7,34 %), Pilar (6,44 %) y Vicente López (5,42 %).
- **`estado`:** 59,13 % nulo y 46 valores distintos. Mezcla estados reales (`excelente` 29,7 %, `bueno` 8,78 %, `a_refaccionar` 2,13 %) con valores numéricos (`0`, `-1`, `-2`), adjetivos de marketing (`luminoso`, `hermosa`, `moderno`) y duplicados por formato (`a_estrenar` / `a estrenar`, `en_obra` / `en obra`).

### 4. Variables numéricas *(EDA 4.1 – 4.2)*
**Precio sin filtrar (4.1):** hay valores imposibles, con máximos de `1.111.111.111` (USD venta) y `1.434.768.228` (ARS venta), y mínimos de `0.06` o `1`. En USD alquiler el promedio es de 18.585 frente a una mediana de 1.600, lo que muestra outliers fuertes.

**Precio filtrado entre P1 y P99** por moneda y operación (4.1b):

| Segmento | Avisos | Mediana | P25 – P75 |
|---|---|---|---|
| USD venta | 577.278 | US$ 135.000 | 80.000 – 240.000 |
| ARS alquiler | 367.201 | $ 650.000 | 500.000 – 900.000 |
| USD alquiler | 244.038 | US$ 1.600 | 1.000 – 2.950 |
| ARS venta | 9.394 | $ 700.000 ⚠️ | 250.000 – 1.400.000 |

- **ARS venta tiene la moneda mal cargada:** una venta con mediana de $700.000 no es realista. Lo más probable es que sean avisos en USD etiquetados como ARS.
- **Cerca del 40 % de los alquileres se publican en USD:** 248.213 frente a 374.379 en ARS (4.1). Cualquier análisis de precio debe separar por moneda.

**Metros cuadrados (4.2):** el máximo es **2.147.483.647**, el valor máximo de un entero (`INT`), que funciona como marcador de error. Por eso el promedio sale distorsionado (9.107 m²), mientras que la **mediana es 78 m² totales y 74 m² cubiertos**.

### 5. Calidad de datos *(EDA 5.1 – 5.3)*
- **`antiguedad`:** el 99,36 % de los registros vale `999`, que funciona como nulo disfrazado. La columna no aporta información.
- **`metros_cuadrados_totales`:** el 4,02 % es nulo o ≤ 0.
- **Duplicados:** 81.114 grupos de URL y precio repetidos, con **130.750 filas extra**. Ejemplo: el mismo aviso de MercadoLibre aparece 12 veces (5.3).

### 6. Precio por zona *(EDA 6.1 – 6.2)*
- El ranking devuelve 99 zonas. **El mismo partido aparece con varios nombres según el portal de origen**, por ejemplo `vicente-lopez`, `bsas-gba-norte/vicente-lopez` y `gba-zona-norte--vicente-lopez`, así que la zona debe normalizarse antes de rankear.
- **Limitación:** este ranking promedia juntos precios en USD y ARS, y de venta y alquiler. El promedio global (340.737) y el orden de las zonas dependen más de la proporción de monedas de cada zona que de su precio real. El siguiente paso es rankear por moneda y tipo de operación usando mediana.

### ✅ Próximos pasos para la capa silver
1. **Tipar** precio, expensas, m², ambientes y piso, y convertir a `null` los valores centinela (`999`, `2147483647`, `NaN`).
2. **Normalizar categóricas:** `tipo_de_operacion`, `moneda`, `estado` y `zona`.
3. **Deduplicar por `url`.**
4. **Eliminar columnas sin información:** `orientacion_*`, `tipo_vendedor` y `antiguedad`.
5. **Marcar outliers** de precio (P1–P99 por moneda y operación) y de m².
6. **Analizar precios** siempre separados por moneda y tipo de operación, con mediana.